In [ ]:
import glob
import cv2
import numpy as np
import voxelmorph as vxm
import tf2onnx
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
pairs = np.load("/home/j/FPGA-project/voxelmorph/data/ixi_pairs.npz")
moving = pairs["moving"]
fixed = pairs["fixed"]
zeros = pairs["zeros"]

print("Loaded pairs:", moving.shape)

In [ ]:
split = int(0.9 * len(moving))
x_train = (moving[:split], fixed[:split])
y_train = [fixed[:split], zeros[:split]]
x_val = (moving[split:], fixed[split:])
y_val = [fixed[split:], zeros[split:]]

print("Train pairs:", x_train[0].shape, "Val pairs:", x_val[0].shape)

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
vol_shape = moving.shape[1:-1] 
nb_features = [[32,32,32,32], [32,32,32,32,32,16]]
vxm_model = vxm.networks.VxmDense(vol_shape, nb_features, int_steps=0)
losses = ['mse', vxm.losses.Grad('l2').loss]
wts = [1.0, 0.01]
vxm_model.compile('adam', loss=losses, loss_weights=wts)

In [ ]:
history = vxm_model.fit(
    x_train, y_train,
    batch_size=8,
    epochs=50,
    validation_data=(x_val, y_val)
)

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs. Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
vxm_model.save('models/vxm2d_model_tf.h5')
vxm_model.save_weights('models/vxm2d_weights_tf.h5')

In [ ]:
spec = (
    tf.TensorSpec((None, *vol_shape, 1), tf.float32, name='moving'),
    tf.TensorSpec((None, *vol_shape, 1), tf.float32, name='fixed'),
)
tf2onnx.convert.from_keras(
    vxm_model, input_signature=spec,
    opset=13, output_path='models/vxm2d_model_tf.onnx'
)

In [ ]:
i = np.random.randint(x_val[0].shape[0])
mov_ex = x_val[0][i:i+1]
fix_ex = x_val[1][i:i+1]

warped, flow = vxm_model.predict([mov_ex, fix_ex])

In [ ]:
u = flow[0, :, :, 0]
v = flow[0, :, :, 1]

step = 8
h, w = u.shape
x = np.arange(0, w, step)
y = np.arange(0, h, step)
X, Y = np.meshgrid(x, y)
U = u[::step, ::step]
V = v[::step, ::step]

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(mov_ex[0, :, :, 0], cmap='gray')
plt.title("Moving")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(fix_ex[0, :, :, 0], cmap='gray')
plt.title("Fixed")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(warped[0, :, :, 0], cmap='gray')
plt.quiver(X, Y, U, V, angles='xy', scale_units='xy', scale=1, color='r')
plt.title("Warped with Flow")
plt.axis('off')

plt.tight_layout()
plt.show()